<a href="https://colab.research.google.com/github/jbundav/prueba/blob/main/informes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Estrategia para la Automatización del Repositorio DSpace

Para automatizar el proceso de preparación de los objetos digitales y sus metadatos, podemos seguir los siguientes pasos:

### Paso 1: Configurar el entorno y las credenciales
Necesitaremos acceso a Google Drive y Google Sheets. Aunque en Colab podemos interactuar directamente con estos servicios, si el proceso se va a replicar o ejecutar en un entorno fuera de Colab, sería importante configurar credenciales para la API de Google (Google Drive API y Google Sheets API).

Para este caso, en Colab podemos usar `gspread` para Google Sheets y `gdown` o las utilidades de `google.colab` para Google Drive.

### Paso 2: Descargar los archivos PDF de Google Drive
Utilizaremos la URL de la carpeta de Google Drive proporcionada para descargar todos los archivos PDF a una carpeta local en tu entorno de Colab.

### Paso 3: Leer los metadatos de Google Sheets
Accederemos a la hoja de cálculo de Google Sheets con los metadatos. La leeremos y la cargaremos en un DataFrame de Pandas para su fácil manipulación.

### Paso 4: Limpiar y transformar los metadatos
El DataFrame de Pandas nos permitirá:
*   Identificar y manejar valores nulos o inconsistencias.
*   Renombrar columnas para que coincidan con el esquema de metadatos de DSpace (Dublin Core, QDC, etc.).
*   Transformar formatos de datos (por ejemplo, fechas, nombres de autores).
*   Asegurarse de que cada entrada de metadatos corresponda a un PDF descargado.

### Paso 5: Asociar metadatos con archivos PDF
Es crucial que cada fila de metadatos en tu CSV final se asocie correctamente con su respectivo archivo PDF. Esto generalmente se logra teniendo un identificador único (como el nombre del archivo PDF) presente tanto en los metadatos como en los nombres de los archivos.

### Paso 6: Generar el archivo CSV de metadatos para DSpace
DSpace generalmente espera un archivo CSV con una columna para el nombre del archivo (del PDF) y columnas para cada campo de metadatos, utilizando un prefijo específico para el esquema (ej. `dc.title`, `dc.creator`, `dc.date.issued`). Crearemos este CSV final a partir del DataFrame procesado.

### Paso 7: Crear la estructura de carpetas y el paquete de depósito
Aunque la solicitud original era solo 'armar una carpeta con pdfs y un csv', DSpace para depósitos masivos (por ejemplo, usando Simple Archive Format) requiere una estructura específica:
*   Una carpeta principal para el paquete de depósito.
*   Dentro de esta, subcarpetas para cada objeto digital.
*   Cada subcarpeta contendrá el archivo PDF (renombrado si es necesario) y un archivo de metadatos XML (dublin_core.xml) o un archivo de contenido (contents).

Para el objetivo actual, nos centraremos en la descarga de PDFs y la creación del CSV con el formato requerido. Si necesitas la estructura de DSpace Simple Archive Format, podemos abordar ese paso adicionalmente.

---

Comencemos con los primeros pasos: descargar los PDFs y leer la hoja de cálculo de metadatos. Necesitaremos instalar algunas bibliotecas.
```

```


In [ ]:
# Instalamos las bibliotecas necesarias
!pip install gdown pandas gspread

## Paso 1.1: Autenticación con Google Drive y Google Sheets
Para acceder a los archivos de Drive y a la hoja de cálculo, necesitaremos autenticarnos. En Colab, la forma más sencilla es usar la funcionalidad de autenticación integrada.

In [ ]:
from google.colab import auth
auth.authenticate_user()

## Paso 2: Descargar los archivos PDF de Google Drive
Usaremos `gdown` para descargar los archivos de la carpeta de Google Drive. Primero, crearemos un directorio para almacenar los PDFs.

In [ ]:
import os
import gdown

# URL de la carpeta de Google Drive (asegúrate de que sea pública o tengas los permisos adecuados)
drive_folder_id = '13JHx0Pr7132D6LFpmKDOmhcqDwspRV0d' # Corregí el ID de la carpeta
output_dir = 'undavcyt_pdfs'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print(f"Descargando archivos PDF de la carpeta de Google Drive ID: {drive_folder_id}...")
try:
    gdown.download_folder(id=drive_folder_id, output=output_dir, use_cookies=False, quiet=False)
    print("Descarga completada. Archivos guardados en: ", output_dir)
    # Verificar si hay PDFs descargados
    pdf_files = [f for f in os.listdir(output_dir) if f.endswith('.pdf')]
    print(f"Se encontraron {len(pdf_files)} archivos PDF en '{output_dir}'.")
except RuntimeError as e:
    print(f"Error al descargar la carpeta de Google Drive: {e}")
    print("Por favor, asegúrate de que la carpeta de Drive tenga permisos de 'Cualquier persona con el enlace puede ver'.")
    print("No se pudo proceder con la descarga de PDFs.")
    pdf_files = [] # Asegurar que pdf_files esté definido aunque falle la descarga

Descargando archivos PDF de la carpeta de Google Drive ID: 13JHx0Pr7132D6LFpmKDOmhcqDwspRV0d...
Descarga completada. Archivos guardados en:  undavcyt_pdfs
Se encontraron 0 archivos PDF en 'undavcyt_pdfs'.


Retrieving folder contents
Failed to retrieve folder contents


## Paso 3: Leer los metadatos de Google Sheets
Ahora leeremos la hoja de cálculo de Google Sheets que contiene los metadatos. Usaremos `gspread` para acceder a la hoja y `pandas` para trabajar con los datos.

In [ ]:
import gspread
import pandas as pd
from google.colab import auth
from google.auth import default

# URL de la hoja de cálculo de Google Sheets
spreadsheet_url = 'https://docs.google.com/spreadsheets/d/1R_dhpJXHFhqAt9ltkthpCQPFis6FtU1AtAmUflxuoT4/edit?gid=869417823#gid=869417823'

# Autenticar con las credenciales de usuario de Colab
try:
    # auth.authenticate_user() ya debería haberse ejecutado. Obtenemos las credenciales por defecto.
    creds, project = default()
    gc = gspread.Client(auth=creds)
    print("Autenticación con gspread exitosa usando credenciales de Colab.")
except Exception as e:
    print(f"Error de autenticación con gspread: {e}")
    print("Asegúrate de que la celda de autenticación de Colab (auth.authenticate_user()) se haya completado correctamente.")
    gc = None # Set gc to None to prevent further errors

if gc:
    try:
        wks = gc.open_by_url(spreadsheet_url)
        # Seleccionar la hoja específica por su gid (869417823)
        worksheet = wks.get_worksheet_by_id(869417823)

        # Leer todos los datos de la hoja en un DataFrame de Pandas
        data = worksheet.get_all_values()
        df_metadata = pd.DataFrame(data[1:], columns=data[0])

        # Mostrar las primeras filas del DataFrame de metadatos
        print("Metadatos cargados del Google Sheet:")
        display(df_metadata.head())
        print(f"Dimensiones del DataFrame: {df_metadata.shape}")
    except Exception as e:
        print(f"Error al abrir o leer la hoja de cálculo: {e}")
        print("Verifica que la URL de la hoja sea correcta, que tengas permisos de acceso y que el gid sea correcto.")
        df_metadata = pd.DataFrame() # Ensure df_metadata is defined even on error
else:
    print("No se pudo autenticar con Google Sheets, no se cargaron los metadatos.")
    df_metadata = pd.DataFrame() # Ensure df_metadata is defined even on error

Autenticación con gspread exitosa usando credenciales de Colab.
Error al abrir o leer la hoja de cálculo: ("Failed to retrieve http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/?recursive=true from the Google Compute Engine metadata service. Status: 404 Response:\nb''", <google.auth.transport.requests._Response object at 0x7d69750e3820>)
Verifica que la URL de la hoja sea correcta, que tengas permisos de acceso y que el gid sea correcto.


## Paso 4, 5 y 6: Limpiar, transformar y asociar metadatos con PDFs y generar el CSV

Basándome en los encabezados de tu Google Sheet (Id, Titulo, Autor/es, Edicion, Fecha, Tipo, Palabras Clave, Resumen, Enlace, PDF, Observaciones), el nombre de la columna que contiene los nombres de los archivos PDF es `PDF`.

In [ ]:
import os

# Aseguramos que output_dir y pdf_files estén disponibles si la celda anterior se ejecutó parcialmente
# (Esto es útil si el usuario ejecuta esta celda directamente después de una interrupción)
if 'output_dir' not in locals():
    output_dir = 'undavcyt_pdfs'
    if not os.path.exists(output_dir):
        print(f"Directorio de PDFs no encontrado: {output_dir}")
        pdf_files = []
    else:
        pdf_files = [f for f in os.listdir(output_dir) if f.endswith('.pdf')]

# Si df_metadata no está definida o está vacía debido a errores anteriores, la inicializamos.
if 'df_metadata' not in locals() or df_metadata.empty:
    print("El DataFrame de metadatos no está disponible o está vacío. Por favor, asegúrate de que la celda de carga de metadatos se ejecutó sin errores.")
    # No podemos continuar sin metadatos, salimos o generamos un df_metadata vacío para evitar errores
    df_metadata = pd.DataFrame()
    # Podemos forzar la salida o pedir al usuario que corrija.
    # Para continuar el flujo, asumimos que el usuario lo corregirá si es el caso.

print("Archivos PDF descargados:")
if pdf_files:
    for f in pdf_files:
        print(f"- {f}")
else:
    print("No se encontraron archivos PDF. Asegúrate de que la descarga de Drive fue exitosa.")

print(f"Total de PDFs: {len(pdf_files)}")

print("\nColumnas de los metadatos cargados:")
if not df_metadata.empty:
    print(df_metadata.columns.tolist())
else:
    print("El DataFrame de metadatos está vacío, no se pueden mostrar las columnas.")

# --- Mapeo de columnas --- (Basado en los encabezados del Google Sheet y lo solicitado)

columna_nombre_archivo_pdf = "PDF" # Columna en el Google Sheet que contiene el nombre del PDF

metadata_mapping = {
    'id': None, # Será generado automáticamente
    'collection': '20.500.13069/1049', # Valor fijo solicitado
    'filename': columna_nombre_archivo_pdf, # Usará el nombre del archivo PDF del Google Sheet
    'dc.title': 'Titulo',
    'dc.creator': 'Autor/es',
    'dc.date.issued': 'Fecha',
    'dc.description': 'Resumen',
    'dc.publisher': None, # No se encontró una columna directa en el sheet
    'dc.subject': 'Palabras Clave',
    'dc.type': 'Tipo',
    'dc.language': None, # No se encontró una columna directa en el sheet
    # Añade más mapeos si tu esquema DSpace lo requiere y tienes columnas correspondientes
}

# --- Procesamiento y generación del CSV ---

if df_metadata.empty or not pdf_files:
    print("No se pueden procesar los metadatos o los PDFs no están disponibles. Asegúrate de que las celdas anteriores se ejecutaron sin errores y que hay datos.")
else:
    # 1. Asegurar que los PDFs en la carpeta tengan una entrada de metadatos
    if columna_nombre_archivo_pdf not in df_metadata.columns:
        print(f"Error: La columna '{columna_nombre_archivo_pdf}' no se encontró en tus metadatos. Por favor, corrige el nombre de la columna en `columna_nombre_archivo_pdf`.")
    else:
        # Convertir la columna de nombres de archivo del DF a strings para asegurar la comparación
        df_metadata[columna_nombre_archivo_pdf] = df_metadata[columna_nombre_archivo_pdf].astype(str)

        # Crear una lista de nombres de archivo de los PDFs descargados (solo el nombre, sin path)
        pdf_basenames = [os.path.basename(f) for f in pdf_files]

        # Filtrar metadatos para incluir solo los registros que tienen un PDF descargado
        df_metadata_filtered = df_metadata[df_metadata[columna_nombre_archivo_pdf].isin(pdf_basenames)].copy()

        if df_metadata_filtered.empty:
            print("Advertencia: No se encontraron coincidencias entre los nombres de archivo PDF descargados y la columna de metadatos especificada.")
            print("Asegúrate de que la columna `columna_nombre_archivo_pdf` y los nombres de los PDFs (incluyendo extensión) coincidan.")
        else:
            print(f"Se encontraron {len(df_metadata_filtered)} registros de metadatos que coinciden con los PDFs descargados.")

            # 2. Preparar el DataFrame final para el CSV de DSpace
            dspace_csv_df = pd.DataFrame()

            # Añadir los campos específicos solicitados: id, collection, filename
            dspace_csv_df['id'] = range(1, len(df_metadata_filtered) + 1) # Generar un ID simple
            dspace_csv_df['collection'] = metadata_mapping['collection']
            # Usamos el nombre del archivo PDF tal cual está en la columna 'PDF' del Sheet.
            dspace_csv_df['filename'] = df_metadata_filtered[metadata_mapping['filename']]

            # Mapear las columnas de metadatos generales a los campos de DSpace
            for dspace_field, original_column in metadata_mapping.items():
                if dspace_field not in ['id', 'collection', 'filename'] and original_column is not None:
                    if original_column in df_metadata_filtered.columns:
                        dspace_csv_df[dspace_field] = df_metadata_filtered[original_column]
                    else:
                        print(f"Advertencia: La columna '{original_column}' para el campo DSpace '{dspace_field}' no se encontró en tus metadatos. Se usará un valor vacío.")
                        dspace_csv_df[dspace_field] = ''
                elif dspace_field not in ['id', 'collection', 'filename'] and original_column is None:
                    dspace_csv_df[dspace_field] = '' # Campos no mapeados se dejan vacíos

            # Asegurarse de que todas las columnas de metadatos esperadas por DSpace estén presentes
            # (esto ya se maneja en el bucle anterior, pero es un buen recordatorio)

            # Exportar a CSV
            output_csv_path = 'dspace_metadata.csv'
            dspace_csv_df.to_csv(output_csv_path, index=False)

            print(f"\nCSV de metadatos generado: {output_csv_path}")
            display(dspace_csv_df.head())

            # Guardamos el df_metadata_filtered para el siguiente paso (SAF)
            global df_metadata_for_saf
            df_metadata_for_saf = df_metadata_filtered
            global dspace_csv_df_final
            dspace_csv_df_final = dspace_csv_df

            print("\nCSV de metadatos listo. Pasando al siguiente paso: Crear la estructura DSpace Simple Archive Format (SAF).")

El DataFrame de metadatos no está disponible o está vacío. Por favor, asegúrate de que la celda de carga de metadatos se ejecutó sin errores.
Archivos PDF descargados:
No se encontraron archivos PDF. Asegúrate de que la descarga de Drive fue exitosa.
Total de PDFs: 0

Columnas de los metadatos cargados:
El DataFrame de metadatos está vacío, no se pueden mostrar las columnas.
No se pueden procesar los metadatos o los PDFs no están disponibles. Asegúrate de que las celdas anteriores se ejecutaron sin errores y que hay datos.


## Paso 7: Crear la estructura DSpace Simple Archive Format (SAF)

Para el formato SAF, DSpace espera una estructura de directorios específica:

```
my_dspace_package/
├── item_001/
│   ├── dublin_core.xml
│   └── file_name.pdf
├── item_002/
│   ├── dublin_core.xml
│   └── another_file.pdf
└── ...
```

Cada 'item_XXX' es un paquete de depósito. Dentro de cada uno, el `dublin_core.xml` contiene los metadatos del ítem, y el archivo PDF es el bitstream. Si hay más archivos adjuntos, se añadirían allí también.

In [ ]:
import shutil
from xml.etree.ElementTree import Element, SubElement, tostring
from xml.dom import minidom

# Directorio base para el paquete SAF
saf_output_dir = 'dspace_saf_package'

# Limpiar el directorio SAF si ya existe
if os.path.exists(saf_output_dir):
    shutil.rmtree(saf_output_dir)
os.makedirs(saf_output_dir)

print(f"Creando estructura SAF en: {saf_output_dir}")

if 'dspace_csv_df_final' not in globals() or dspace_csv_df_final.empty:
    print("Error: El DataFrame final de CSV no está disponible. No se puede crear el SAF.")
else:
    for index, row in dspace_csv_df_final.iterrows():
        item_id = row['id']
        item_folder = os.path.join(saf_output_dir, f"item_{item_id:03d}") # item_001, item_002, etc.
        os.makedirs(item_folder)

        # --- Crear dublin_core.xml ---
        # DSpace espera un XML que contiene los metadatos Dublin Core
        dc = Element('dublin_core')
        dc.set('schema', 'dc') # Aseguramos el schema

        # Añadir metadatos al XML
        for col_name, value in row.items():
            if col_name.startswith('dc.'): # Solo procesar columnas que son metadatos DC
                parts = col_name.split('.')
                element_name = parts[1]
                qualifier_name = parts[2] if len(parts) > 2 else None

                if value and str(value).strip(): # Solo añadir si hay valor y no está vacío
                    dcvalue = SubElement(dc, 'dcvalue')
                    dcvalue.set('element', element_name)
                    if qualifier_name:
                        dcvalue.set('qualifier', qualifier_name)
                    dcvalue.set('language', 'es') # Asumimos idioma español por defecto, si no hay columna de idioma
                    dcvalue.text = str(value)

        # Añadir 'id' y 'collection' como metadatos si se desea, o manejarlos de otra forma en DSpace
        # Para este ejemplo, solo los metadatos 'dc.*' van al dublin_core.xml

        # Bonificar el XML para una mejor legibilidad
        rough_string = tostring(dc, 'utf-8')
        reparsed = minidom.parseString(rough_string)
        with open(os.path.join(item_folder, 'dublin_core.xml'), 'w', encoding='utf-8') as f:
            f.write(reparsed.toprettyxml(indent="  "))

        # --- Copiar el archivo PDF ---
        original_pdf_filename = row['filename']
        source_pdf_path = os.path.join(output_dir, original_pdf_filename)
        destination_pdf_path = os.path.join(item_folder, original_pdf_filename)

        if os.path.exists(source_pdf_path):
            shutil.copy(source_pdf_path, destination_pdf_path)
            # Crear el archivo 'contents' que lista los bitstreams
            with open(os.path.join(item_folder, 'contents'), 'w', encoding='utf-8') as f:
                f.write(original_pdf_filename + "\n")
        else:
            print(f"Advertencia: El archivo PDF '{original_pdf_filename}' no se encontró en '{output_dir}'. No se copió a SAF.")

    print(f"\nCreación del paquete SAF completada en '{saf_output_dir}'.")
    print("Puedes comprimir esta carpeta (por ejemplo, `tar -czvf dspace_saf_package.tar.gz dspace_saf_package/`) para subirla a DSpace.")

Creando estructura SAF en: dspace_saf_package
Error: El DataFrame final de CSV no está disponible. No se puede crear el SAF.


## Paso 4, 5 y 6: Limpiar, transformar y asociar metadatos con PDFs y generar el CSV

Primero, listemos los archivos PDF que se descargaron y mostremos las columnas de los metadatos para identificar la columna clave de asociación.

In [ ]:
import os

# Listar los archivos PDF descargados (usamos 'pdf_files' de la celda anterior de descarga)
# Si la celda de descarga falló, esta lista podría estar vacía o no definida. Por eso es importante que funcione.

# Aseguramos que output_dir y pdf_files estén disponibles si la celda anterior se ejecutó parcialmente
if 'output_dir' not in locals():
    output_dir = 'undavcyt_pdfs'
    if not os.path.exists(output_dir):
        print(f"Directorio de PDFs no encontrado: {output_dir}")
        pdf_files = []
    else:
        pdf_files = [f for f in os.listdir(output_dir) if f.endswith('.pdf')]

print("Archivos PDF descargados:")
if pdf_files:
    for f in pdf_files:
        print(f"- {f}")
else:
    print("No se encontraron archivos PDF. Asegúrate de que la descarga de Drive fue exitosa.")

print(f"Total de PDFs: {len(pdf_files)}")

print("\nColumnas de los metadatos cargados:")
if not df_metadata.empty:
    print(df_metadata.columns.tolist())
else:
    print("El DataFrame de metadatos está vacío. Por favor, revisa la celda de carga de metadatos.")

# Pedir al usuario que identifique la columna del DataFrame de metadatos que contiene el nombre del archivo PDF.
print("\nPara continuar, necesito que me indiques cuál de las columnas de metadatos (`df_metadata.columns`) contiene el nombre exacto del archivo PDF asociado a cada registro. Por favor, introduce el nombre exacto de la columna.")
# Esta variable será llenada manualmente por el usuario o en un siguiente turno. Por ahora la dejamos como None o una cadena vacía.
# columna_nombre_archivo_pdf = None # Por ahora, se espera que el usuario la defina.

Archivos PDF descargados:
No se encontraron archivos PDF. Asegúrate de que la descarga de Drive fue exitosa.
Total de PDFs: 0

Columnas de los metadatos cargados:
El DataFrame de metadatos está vacío. Por favor, revisa la celda de carga de metadatos.

Para continuar, necesito que me indiques cuál de las columnas de metadatos (`df_metadata.columns`) contiene el nombre exacto del archivo PDF asociado a cada registro. Por favor, introduce el nombre exacto de la columna.


Una vez que hayas identificado la columna que contiene el nombre del archivo PDF, asigna su nombre a la variable `columna_nombre_archivo_pdf` en la siguiente celda. También deberás asignar los nombres de las columnas que se corresponderán a los campos `dc.title`, `dc.creator`, `dc.date.issued`, `dc.description`, `dc.publisher`, `dc.subject`, `dc.type` y `dc.language` del esquema Dublin Core. Aquí hay un ejemplo de cómo podrías asignar las columnas.

In [ ]:
# Asigna el nombre de la columna que contiene el nombre del archivo PDF
columna_nombre_archivo_pdf = "nombre_archivo_pdf" # <-- **MODIFICA ESTO** con el nombre correcto de tu columna

# Asigna las columnas de tu DataFrame a los campos Dublin Core de DSpace
# Modifica estos nombres de columna para que coincidan con los de tu df_metadata

metadata_mapping = {
    'id': None, # Será generado automáticamente o puedes mapearlo a una columna de ID en tu Sheet si existe
    'collection': '20.500.13069/1049', # Valor fijo solicitado
    'filename': columna_nombre_archivo_pdf, # Usará el nombre del archivo PDF
    'dc.title': 'titulo', # EJEMPLO: Columna para el título
    'dc.creator': 'autor', # EJEMPLO: Columna para el/los autor/es
    'dc.date.issued': 'fecha_publicacion', # EJEMPLO: Columna para la fecha de publicación
    'dc.description': 'resumen', # EJEMPLO: Columna para la descripción/resumen
    'dc.publisher': 'editorial', # EJEMPLO: Columna para el editor
    'dc.subject': 'palabras_clave', # EJEMPLO: Columna para las palabras clave
    'dc.type': 'tipo_documento', # EJEMPLO: Columna para el tipo de documento
    'dc.language': 'idioma', # EJEMPLO: Columna para el idioma
    # Añade más mapeos según necesites para tu esquema de DSpace
}

# --- Procesamiento y generación del CSV ---

if df_metadata.empty or not pdf_files:
    print("No se pueden procesar los metadatos o los PDFs no están disponibles. Asegúrate de que las celdas anteriores se ejecutaron sin errores.")
else:
    # 1. Asegurar que los PDFs en la carpeta tengan una entrada de metadatos
    #    Aquí asumimos que el 'columna_nombre_archivo_pdf' contiene el nombre base del PDF (e.g., 'informe.pdf')
    #    y lo usaremos para filtrar el DataFrame.

    # Asegurarse de que la columna existe en el DataFrame
    if columna_nombre_archivo_pdf not in df_metadata.columns:
        print(f"Error: La columna '{columna_nombre_archivo_pdf}' no se encontró en tus metadatos. Por favor, corrige el nombre de la columna.")
    else:
        df_metadata_filtered = df_metadata[df_metadata[columna_nombre_archivo_pdf].isin(pdf_files)].copy()

        if df_metadata_filtered.empty:
            print("Advertencia: No se encontraron coincidencias entre los nombres de archivo PDF descargados y la columna de metadatos especificada.")
            print("Asegúrate de que la columna `columna_nombre_archivo_pdf` y los nombres de los PDFs (incluyendo extensión) coincidan.")
        else:
            print(f"Se encontraron {len(df_metadata_filtered)} registros de metadatos que coinciden con los PDFs descargados.")

            # 2. Preparar el DataFrame final para el CSV de DSpace
            dspace_csv_df = pd.DataFrame()

            # Añadir los campos específicos solicitados: id, collection, filename
            dspace_csv_df['id'] = range(1, len(df_metadata_filtered) + 1) # Generar un ID simple, o puedes usar una columna de tu sheet si existe
            dspace_csv_df['collection'] = metadata_mapping['collection']
            dspace_csv_df['filename'] = df_metadata_filtered[metadata_mapping['filename']]

            # Mapear las columnas de metadatos generales a los campos de DSpace
            for dspace_field, original_column in metadata_mapping.items():
                if dspace_field not in ['id', 'collection', 'filename'] and original_column is not None:
                    if original_column in df_metadata_filtered.columns:
                        dspace_csv_df[dspace_field] = df_metadata_filtered[original_column]
                    else:
                        print(f"Advertencia: La columna '{original_column}' para el campo DSpace '{dspace_field}' no se encontró en tus metadatos. Se usará un valor vacío.")
                        dspace_csv_df[dspace_field] = ''

            # Asegurarse de que todas las columnas de metadatos esperadas por DSpace estén presentes,
            # incluso si no se mapearon directamente, inicializándolas vacías si es necesario.
            # Esto es un ejemplo, se debería ajustar al esquema exacto de DSpace.
            required_dspace_fields = [
                'dc.title', 'dc.creator', 'dc.date.issued', 'dc.description',
                'dc.publisher', 'dc.subject', 'dc.type', 'dc.language'
            ]
            for field in required_dspace_fields:
                if field not in dspace_csv_df.columns:
                    dspace_csv_df[field] = ''

            # Exportar a CSV
            output_csv_path = 'dspace_metadata.csv'
            dspace_csv_df.to_csv(output_csv_path, index=False)

            print(f"\nCSV de metadatos generado: {output_csv_path}")
            display(dspace_csv_df.head())

            # Ahora, el siguiente paso sería crear la estructura SAF
            print("\nEl siguiente paso es crear la estructura de DSpace Simple Archive Format (SAF).")

No se pueden procesar los metadatos o los PDFs no están disponibles. Asegúrate de que las celdas anteriores se ejecutaron sin errores.
